In [4]:
import os
import json
import re

In [5]:
#PATH_AUDITS = './../datasets/production/audits.json'
#PATH_ENTITIES = './../datasets/niort/entities.json'
#PATH_USERS = './../datasets/production/users.json'

PATH_AUDITS = './../datasets/data-import/audits.json'
PATH_ENTITIES = './../datasets/data-import/entities.json'
PATH_USERS = './../datasets/production/users.json'

In [6]:
# Load the PATH_AUDITS json into dict
entities = {}

with open(PATH_ENTITIES, 'r') as file:
    entities = json.load(file)

entity_list = []
for entity in entities:
    entity_list.append(entity['id'])
print(f'Loaded {len(entities)} entities from {PATH_ENTITIES}')


Loaded 15018 entities from ./../datasets/data-import/entities.json


In [7]:
# Load the PATH_AUDITS json into dict
audits = {}
with open(PATH_AUDITS, 'r') as file:
    audits = json.load(file)

print(f'Loaded {len(audits)} audits from {PATH_AUDITS}')

Loaded 22929 audits from ./../datasets/data-import/audits.json


In [9]:
# aggregate audits by entity id
audits_by_entity = {}
for ai, audit in enumerate(audits):
    entity_id = audit.get('entityId')
    if entity_id and entity_id in entity_list:
        if entity_id not in audits_by_entity:
            audits_by_entity[entity_id] = []
        audits_by_entity[entity_id].append(audit)
    else:        
        print(f'Entity {entity_id} not in entity_list')

print(f'Aggregated audits by entity, found {len(audits_by_entity)} entities with audits')

Aggregated audits by entity, found 9111 entities with audits


In [10]:
# iterate over entities, find all items in audits for the entityId and fill the
#  audits_by_user dictionary so the first key level is user, 
#  second key level is entityId, for each entityId on second level store the relevant audit item count, 
#  on top level, user should store the total count of audits

audits_by_user = {}
for entity_id, entity_audits in audits_by_entity.items():
    for audit in entity_audits:
        user_id = audit.get('user')
        if user_id:
            if user_id not in audits_by_user:
                audits_by_user[user_id] = {}
            audits_by_user[user_id][entity_id] = audits_by_user[user_id].get(entity_id, 0) + 1
        audits_by_user[user_id]['total'] = audits_by_user[user_id].get('total', 0) + 1
print(f'Aggregated audits by user, found {len(audits_by_user)} users with audits')

Aggregated audits by user, found 10 users with audits


In [11]:
# load the PATH_USERS json into dict where userId is the key

users = {}
with open(PATH_USERS, 'r') as file:
    users = json.load(file)

# Transform users list into a dictionary with userId as the key
users = {user['id']: user for user in users}
print(f'Loaded {len(users)} users from {PATH_USERS}')

Loaded 19 users from ./../datasets/production/users.json


In [ ]:
# For each user in audits_by_user, find the user in users and add the user data to the audits_by_user dictionary
for user_id, user_audits in audits_by_user.items():
    user = users[user_id]
    if user:
        audits_by_user[user_id]['user'] = {
            'id': user_id,
            'name': user.get('name'),
            'email': user.get('email'),
            'role': user.get('role')
        }
    else:
        audits_by_user[user_id]['user'] = {
            'id': user_id,
            'name': None,
            'email': None,
            'role': None
        } 
print(f'Added user data to audits_by_user, now contains {len(audits_by_user)} users with audits')

Added user data to audits_by_user, now contains 10 users with audits


In [13]:
# calculate percentage for each user
total_audits = sum(user_data.get('total', 0) for user_data in audits_by_user.values())
for user_id, user_data in audits_by_user.items():
    user_data['percentage'] = (user_data.get('total', 0) / total_audits * 100) if total_audits > 0 else 0

for user in audits_by_user:
  print(f'User {user} {audits_by_user[user]["user"]["name"]} has {audits_by_user[user].get("total", 0)} audits ({audits_by_user[user]["percentage"]:.2f}%)')

User 101 Robert Shaw has 4469 audits (19.49%)
User 100 David Zbíral has 8972 audits (39.13%)
User 103 Katia Riccardo has 4052 audits (17.67%)
User 102 Davor Salihovic has 1296 audits (5.65%)
User 02868c93-517e-4602-94b7-057ce1a393fd Katalin Suba has 3425 audits (14.94%)
User 1 admin has 219 audits (0.96%)
User 107 Larissa de Freitas Lyth has 210 audits (0.92%)
User 5a8b6aff-3dc1-40b2-9704-82b47a952d4c stanislaw.banach has 274 audits (1.19%)
User 151 Tomáš Hampejs has 7 audits (0.03%)
User 105 Reima Välimäki has 5 audits (0.02%)
